# 1. Environment Setup & Reproducibility
Install required packages, configure random seeds across all libraries for deterministic execution, and configure the compute device.

In [2]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(2025)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 62.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.6 MB/s eta 0:00:00
Configured device: cuda


# 2. Ontology Knowledge Graph Engine
Initializes the `OntologyEngine` to extract continuous 24-dimensional feature vectors representing emotions, cognitive appraisals, intensity, polarity, and negation cues.

In [3]:
# Initialize OntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc_ekman.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = OntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.85),
    manual_boost=onto_cfg.get('manual_boost', 1.2),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.5),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Found ontology file at: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loading Knowledge Graph: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loaded 90420 triples.
Lexicon initialized with 2540 entries.


# 3. Lexical Coverage Analysis & Diagnostic Suite
Computes feature hit rates and active dimensions across datasets and verifies the engine against core linguistic diagnostic test cases.

In [4]:
# 1. Diagnostic test suite
test_cases = [
    ("tôi không hề thích nó", "Negation Scope Check"),
    ("cảm thấy không vui chút nào", "Flip Logic Check"),
    ("bún chả rất ngon", "Ambiguity Filter Check (Chả)"),
    ("làm hay nghỉ", "Ambiguity Filter Check (Hay)"),
    ("thất vọng tràn trề", "Multi-Word Expression Recognition"),
]

print("==================== ONTOLOGY DIAGNOSTIC REPORT ====================")
for text, desc in test_cases:
    vec, tokens, trace = ontengine.getvector(text, debug=True)
    matched = [t["phrase"] for t in trace] if trace else ["None"]
    is_neg = any(t["data"]["isnegation"] for t in trace) if trace else False

    labels = ontengine.ALLEMOTIONS
    emotions_vals = vec[:7]
    max_idx = np.argmax(emotions_vals)
    dominant = (
        f"{labels[max_idx]} ({emotions_vals[max_idx]:.4f})"
        if emotions_vals[max_idx] > 0
        else "Neutral/None"
    )

    print(f"\nScenario: {desc}")
    print(f"  - Input: '{text}'")
    print(f"  - Matched Phrases: {matched}")
    print(f"  - Negation Detected: {is_neg}")
    print(f"  - Dominant Emotion: {dominant}")
print("\n====================================================================")

==================== ONTOLOGY DIAGNOSTIC REPORT ====================

Scenario: Negation Scope Check
  - Input: 'tôi không hề thích nó'
  - Matched Phrases: ['không', 'thích']
  - Negation Detected: True
  - Dominant Emotion: Happiness (0.4510)

Scenario: Flip Logic Check
  - Input: 'cảm thấy không vui chút nào'
  - Matched Phrases: ['không', 'vui']
  - Negation Detected: True
  - Dominant Emotion: Happiness (0.4941)

Scenario: Ambiguity Filter Check (Chả)
  - Input: 'bún chả rất ngon'
  - Matched Phrases: ['rất']
  - Negation Detected: False
  - Dominant Emotion: Happiness (0.4189)

Scenario: Ambiguity Filter Check (Hay)
  - Input: 'làm hay nghỉ'
  - Matched Phrases: ['nghỉ']
  - Negation Detected: False
  - Dominant Emotion: Sadness (0.3888)

Scenario: Multi-Word Expression Recognition
  - Input: 'thất vọng tràn trề'
  - Matched Phrases: ['thất vọng']
  - Negation Detected: False
  - Dominant Emotion: Sadness (0.5764)



# 4. Dataset Loading, Preprocessing & DataLoaders
Loads the `vsfcekman-1` train, dev, and test sets, encodes the 7 Ekman emotion classes, extracts ontology features, and constructs PyTorch DataLoaders with balanced class weights.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")


class EmotionDataset(Dataset):

    def __init__(self, df, tokenizer, ontology_engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.ont_engine = ontology_engine
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        raw_text = str(self.df.iloc[index]["Sentence"])
        label = self.df.iloc[index]["label_idx"]

        encoding = self.tokenizer(
            raw_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        ont_vector = self.ont_engine.getvector(raw_text)
        if isinstance(ont_vector, tuple):
            ont_vector = ont_vector[0]

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "ontology_features": torch.tensor(ont_vector, dtype=torch.float),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# Dataset paths
BASE_DATA_PATH = "../../data/vsfc-ekman"

df_train = pd.read_csv(f"{BASE_DATA_PATH}/train.csv")
df_valid = pd.read_csv(f"{BASE_DATA_PATH}/dev.csv")
df_test = pd.read_csv(f"{BASE_DATA_PATH}/test.csv")

for df in [df_train, df_valid, df_test]:
    df.rename(
        columns={"Text": "Sentence", "Ekman_Label": "Emotion"}, inplace=True
    )
    df.dropna(subset=["Sentence", "Emotion"], inplace=True)

print(
    f"Loaded datasets: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}"
)

# Label Encoding
le = LabelEncoder()
all_labels = sorted(df_train["Emotion"].unique().astype(str))
le.fit(all_labels)
print(f"Classes ({len(le.classes_)}): {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df["label_idx"] = le.transform(df["Emotion"].astype(str))

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(
    EmotionDataset(df_train, tokenizer, ontengine),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader = DataLoader(
    EmotionDataset(df_valid, tokenizer, ontengine), batch_size=BATCH_SIZE
)
test_loader = DataLoader(
    EmotionDataset(df_test, tokenizer, ontengine), batch_size=BATCH_SIZE
)

# Compute balanced class weights
train_labels = df_train["label_idx"].tolist()
class_weights = compute_class_weight(
    "balanced", classes=np.unique(train_labels), y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights: {weights_tensor.cpu().numpy()}")

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

Loaded datasets: Train=11426, Dev=1583, Test=3166
Classes (7): ['Anger' 'Disgust' 'Fear' 'Happiness' 'Neutral' 'Sadness' 'Surprise']
Class Weights: [ 0.76168257  2.6327188  41.85348     0.28987494  3.571741    0.65448505
 38.863945  ]


# 5. Model Architectures
Defines neural-symbolic architectures for 7-class emotion classification:
1. `ViSoBERT_Fusion`: Standard ViSoBERT backbone supporting baseline, concatenation, and gated projection.
2. `ViSoBERT_Residual_Fusion`: Multi-layer ontology projector with gated residual connections and LayerNorm.
3. `OntologyConceptProjector` & `ViSoBERT_Ontology_CrossAttention` (OCA): Token-level multi-head cross-attention over 5 projected ontology concept tokens.

In [6]:
# Import ViSoBERT fusion & cross-attention architectures from src.models
from src.models.visobert_fusion_models import (
    ViSoBERT_Fusion,
    ViSoBERT_DeepOntology,
    ViSoBERT_DeepOntology_V2,
    ViSoBERT_Residual_Fusion,
    OntologyConceptProjector,
    ViSoBERT_Ontology_CrossAttention,
)

ONT_INPUT_DIM = 24
print('✅ Models loaded from src.models.visobert_fusion_models')


# 6. Training Pipeline & Evaluation Utilities
Defines the training loop with validation early stopping and best checkpoint restoration.

In [7]:
def evaluate(model, loader):
    """Compute accuracy and macro F1 score on a DataLoader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(ids, mask, ont)
            _, pr = torch.max(outputs, dim=1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(model, name, epochs=15, patience=3):
    """Execute model training with weighted CrossEntropyLoss and early stopping."""
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(ids, mask, ont)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_acc, val_f1 = evaluate(model, valid_loader)
        avg_loss = total_loss / len(train_loader)
        print(
            f"Epoch {epoch + 1:02d}/{epochs:02d} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}",
            end=" ",
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            patience_counter = 0
            print("(New Best)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)

    return model

# 7. Model Training Across Configurations
Trains baseline and comparative neural-symbolic models, including Gated Residual Fusion and Ontology-Guided Cross-Attention (OCA).

In [8]:
NUM_CLASSES = len(le.classes_)

# 1. Baseline
m1 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="none")
m1 = train_experiment(m1, "1. Baseline (ViSoBERT)", epochs=15, patience=3)

# 2. Raw + Concat
m2 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="concat", ontology_dim=None)
m2 = train_experiment(m2, "2. Raw (24d) + Concat", epochs=15, patience=3)

# 3. Raw + Gate
m3 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="gate", ontology_dim=None)
m3 = train_experiment(m3, "3. Raw (24d) + Gate", epochs=15, patience=3)

# 4. Dense + Concat
m4 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="concat", ontology_dim=64)
m4 = train_experiment(m4, "4. Dense (64d) + Concat", epochs=15, patience=3)

# 5. Dense + Gate
m5 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="gate", ontology_dim=64)
m5 = train_experiment(m5, "5. Dense (64d) + Gate", epochs=15, patience=3)

# 6. Gated Residual Fusion
m_res = ViSoBERT_Residual_Fusion(
    n_classes=NUM_CLASSES, ont_input_dim=ONT_INPUT_DIM
)
m_res = train_experiment(
    m_res, "6. Gated Residual Fusion", epochs=15, patience=3
)

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. Baseline (ViSoBERT)


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 5. Dense (64d) + Gate
Epoch 01/15 | Loss: 1.5225 | Val F1: 0.4797 | Val Acc: 0.6886 (New Best)
Epoch 02/15 | Loss: 1.0678 | Val F1: 0.4374 | Val Acc: 0.6292 (Patience: 1/3)
Epoch 03/15 | Loss: 0.7597 | Val F1: 0.4596 | Val Acc: 0.6665 (Patience: 2/3)
Epoch 04/15 | Loss: 0.5485 | Val F1: 0.5347 | Val Acc: 0.7359 (New Best)
Epoch 05/15 | Loss: 0.4171 | Val F1: 0.5497 | Val Acc: 0.7581 (New Best)
Epoch 06/15 | Loss: 0.3133 | Val F1: 0.5389 | Val Acc: 0.7505 (Patience: 1/3)
Epoch 07/15 | Loss: 0.2300 | Val F1: 0.5632 | Val Acc: 0.7625 (New Best)
Epoch 08/15 | Loss: 0.1664 | Val F1: 0.5626 | Val Acc: 0.7656 (Patience: 1/3)
Epoch 09/15 | Loss: 0.1502 | Val F1: 0.5771 | Val Acc: 0.7808 (New Best)
Epoch 10/15 | Loss: 0.1224 | Val F1: 0.5356 | Val Acc: 0.7764 (Patience: 1/3)
Epoch 11/15 | Loss: 0.0657 | Val F1: 0.5581 | Val Acc: 0.7909 (Patience: 2/3)
Epoch 12/15 | Loss: 0.0662 | Val F1: 0.5453 | Val Acc: 0.7688 (Patience: 3/3)
Early stopping triggered after 12 epochs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 6. Gated Residual Fusion
Epoch 01/15 | Loss: 1.5500 | Val F1: 0.4244 | Val Acc: 0.6677 (New Best)
Epoch 02/15 | Loss: 1.0942 | Val F1: 0.5098 | Val Acc: 0.7486 (New Best)
Epoch 03/15 | Loss: 0.7899 | Val F1: 0.5459 | Val Acc: 0.7549 (New Best)
Epoch 04/15 | Loss: 0.5640 | Val F1: 0.5777 | Val Acc: 0.7745 (New Best)
Epoch 05/15 | Loss: 0.4331 | Val F1: 0.5857 | Val Acc: 0.7764 (New Best)
Epoch 06/15 | Loss: 0.3228 | Val F1: 0.5642 | Val Acc: 0.7587 (Patience: 1/3)
Epoch 07/15 | Loss: 0.2375 | Val F1: 0.5791 | Val Acc: 0.7846 (Patience: 2/3)
Epoch 08/15 | Loss: 0.1927 | Val F1: 0.5776 | Val Acc: 0.7903 (Patience: 3/3)
Early stopping triggered after 8 epochs.


# 8. Final Benchmark Evaluation & Classification Reports
Collects model predictions on the test set and outputs macro F1, overall accuracy, and class-level classification metrics.

In [9]:
def get_predictions(model, loader):
    """Extract ground truth and model predictions on a dataset loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(ids, mask, ont)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# Collect test predictions
y_true, p1 = get_predictions(m1, test_loader)
_, p2 = get_predictions(m2, test_loader)
_, p3 = get_predictions(m3, test_loader)
_, p4 = get_predictions(m4, test_loader)
_, p5 = get_predictions(m5, test_loader)
_, p_res = get_predictions(m_res, test_loader)

models_list = [
    ("Baseline (ViSoBERT)", "-", p1),
    ("Raw + Concat", "24 (Raw)", p2),
    ("Raw + Gate", "24 (Raw)", p3),
    ("Dense + Concat", "64 (Dense)", p4),
    ("Dense + Gate", "64 (Dense)", p5),
    ("Gated Residual Fusion", "768 (Projected)", p_res),
]

summary_records = []
for name, dim, preds in models_list:
    summary_records.append(
        {
            "Model Architecture": name,
            "Ontology Dim": dim,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n==================== TEST SET BENCHMARK RESULTS ====================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print("\n========== CLASSIFICATION REPORT: GATED RESIDUAL FUSION ==========")
print(classification_report(y_true, p_res, target_names=le.classes_, digits=4))


==================== TEST SET BENCHMARK RESULTS ====================
   Model Architecture    Ontology Dim  Accuracy  Macro F1
  Baseline (ViSoBERT)               -    0.7720    0.6000
         Raw + Concat        24 (Raw)    0.7682    0.5657
           Raw + Gate        24 (Raw)    0.7618    0.5922
       Dense + Concat      64 (Dense)    0.7647    0.5705
         Dense + Gate      64 (Dense)    0.7669    0.6153
Gated Residual Fusion 768 (Projected)    0.7473    0.5560

========== CLASSIFICATION REPORT: GATED RESIDUAL FUSION ==========
              precision    recall  f1-score   support

       Anger     0.6474    0.6788    0.6627       576
     Disgust     0.3163    0.4526    0.3724       137
        Fear     0.2632    0.3846    0.3125        13
   Happiness     0.9371    0.9023    0.9194      1586
     Neutral     0.3496    0.5150    0.4165       167
     Sadness     0.6844    0.5676    0.6206       680
    Surprise     0.5000    0.7143    0.5882         7

    accuracy          

# 9. Neuro-Symbolic Rule-Based Post-Processing
Evaluates an override reasoning mechanism applying high-confidence ontology rules over uncertain baseline model outputs.

In [10]:
class NeuroSymbolicReasoner:
    """Combines neural probability outputs with symbolic ontology rules."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        le_classes,
        ont_threshold=0.6,
        model_conf_limit=0.9,
    ):
        self.model = model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.classes = le_classes
        self.cls_to_idx = {name: i for i, name in enumerate(le_classes)}
        self.ont_threshold = ont_threshold
        self.model_conf_limit = model_conf_limit

    def infer(self, text):
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec = self.engine.getvector(text)
        if isinstance(vec, tuple):
            vec = vec[0]
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        pred_label_str = self.classes[pred_idx]

        final_idx = pred_idx

        # Rule: If prediction is Neutral with moderate confidence, check for strong emotion hits
        if (
            pred_label_str == "Neutral"
            and confidence < self.model_conf_limit
        ):
            ont_emotions = vec[:7]
            max_ont_idx = np.argmax(ont_emotions)
            max_ont_val = ont_emotions[max_ont_idx]
            max_ont_label = self.engine.ALLEMOTIONS[max_ont_idx]

            if max_ont_val > self.ont_threshold and max_ont_label != "Neutral":
                if max_ont_label in self.cls_to_idx:
                    final_idx = self.cls_to_idx[max_ont_label]

        return pred_idx, final_idx


reasoner = NeuroSymbolicReasoner(
    model=m1,
    engine=ontengine,
    tokenizer=tokenizer,
    device=device,
    le_classes=le.classes_,
    ont_threshold=0.6,
    model_conf_limit=0.9,
)

y_base = []
y_rule = []

for txt in tqdm(df_test["Sentence"], desc="Rule Post-processing"):
    p_b, p_r = reasoner.infer(txt)
    y_base.append(p_b)
    y_rule.append(p_r)

acc_base = accuracy_score(y_true, y_base)
f1_base = f1_score(y_true, y_base, average="macro")
acc_rule = accuracy_score(y_true, y_rule)
f1_rule = f1_score(y_true, y_rule, average="macro")

df_rule_comparison = pd.DataFrame(
    {
        "Method": ["Baseline (m1)", "Baseline + Ontology Rules"],
        "Accuracy": [acc_base, acc_rule],
        "Macro F1": [f1_base, f1_rule],
        "Delta": [0.0, f1_rule - f1_base],
    }
)

print("\n============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============")
print(df_rule_comparison.round(4).to_string(index=False))

total_overrides = sum(1 for a, b in zip(y_base, y_rule) if a != b)
print(
    f"\nTotal samples overridden: {total_overrides}/{len(df_test)} ({total_overrides / len(df_test) * 100:.2f}%)"
)

Rule Post-processing:   0%|          | 0/3166 [00:00<?, ?it/s]


============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============
                   Method  Accuracy  Macro F1   Delta
            Baseline (m1)    0.7720    0.6000  0.0000
Baseline + Ontology Rules    0.7716    0.5992 -0.0009

Total samples overridden: 1/3166 (0.03%)


# 10. Statistical Significance Testing (Paired Permutation Test)
Calculates exact permutation p-values over 10,000 resamples to evaluate statistical significance ($p < 0.05$) between model prediction vectors.

In [11]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    ("m1 (Baseline) vs m3 (Raw+Gate)", p1, p3),
    ("m1 (Baseline) vs m_res (Residual)", p1, p_res),
    ("m3 (Raw+Gate) vs m_res (Residual)", p3, p_res),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

results = []
for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    results.append(
        {
            "Comparison": name,
            "F1 (Model A)": round(f1_a, 4),
            "F1 (Model B)": round(f1_b, 4),
            "Diff (F1)": round(diff_f1, 4),
            "p-val (F1)": round(p_val_f1, 5),
            "Sig (F1)": sig_f1,
            "p-val (Acc)": round(p_val_acc, 5),
            "Sig (Acc)": sig_acc,
        }
    )

df_perm = pd.DataFrame(results)
print(df_perm.to_string(index=False))
print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============
                       Comparison  F1 (Model A)  F1 (Model B)  Diff (F1)  p-val (F1)               Sig (F1)  p-val (Acc)              Sig (Acc)
   m1 (Baseline) vs m3 (Raw+Gate)        0.6000        0.5922     0.0079     0.54955        Not Significant      0.12019        Not Significant
m1 (Baseline) vs m_res (Residual)        0.6000        0.5560     0.0440     0.03820 Significant (p < 0.05)      0.00050 Significant (p < 0.05)
m3 (Raw+Gate) vs m_res (Residual)        0.5922        0.5560     0.0361     0.08029        Not Significant      0.02300 Significant (p < 0.05)

